In [ ]:
from pathlib import Path
import subprocess, sys
script = r'''#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
train.py — entrenamiento reforzado para clasificación REAL vs IA en Kaggle.

Uso en una celda Kaggle:
    !python /kaggle/working/train.py

También se puede crear y ejecutar desde una celda:
    %%writefile /kaggle/working/train.py
    [contenido de este archivo]

Convención interna: 0 = REAL, 1 = IA.
IMPORTANTE: revisa LABEL_MAPS si las etiquetas originales de tus datasets difieren.
"""

import os
# Evita algunos problemas de paralelismo entre bibliotecas numéricas.
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import gc
import io
import json
import math
import random
import time
import hashlib
import warnings
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image, ImageFile, ImageOps
from datasets import load_dataset, interleave_datasets
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, average_precision_score,
    precision_score, recall_score, confusion_matrix, classification_report
)
from tqdm.auto import tqdm

# ============================ CONFIG ============================
MODEL_NAME = "imager-1-tiny"
WORK_DIR = Path("/kaggle/working")
CHECKPOINT_DIR = WORK_DIR / "checkpoints_tiny"
MODEL_DIR = WORK_DIR / "models_tiny"
LOG_DIR = WORK_DIR / "logs_tiny"
for _p in (CHECKPOINT_DIR, MODEL_DIR, LOG_DIR):
    _p.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = CHECKPOINT_DIR / "latest.pth"
BEST_PATH = MODEL_DIR / f"{MODEL_NAME}_best.pth"
FINAL_PATH = WORK_DIR / "train.py_model_final.pth"
METRICS_PATH = LOG_DIR / "metrics.jsonl"
CONFIG_PATH = LOG_DIR / "config.json"

IMAGE_SIZE = int(os.getenv("IMAGE_SIZE", "224"))
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "64"))
NUM_WORKERS = int(os.getenv("NUM_WORKERS", "2"))
MAX_RUNTIME_SECONDS = int(float(os.getenv("MAX_RUNTIME_HOURS", "11.5")) * 3600)
CHECKPOINT_EVERY_STEPS = int(os.getenv("CHECKPOINT_EVERY_STEPS", "500"))
EPOCHS = int(os.getenv("EPOCHS", "100"))
VAL_FRACTION = float(os.getenv("VAL_FRACTION", "0.15"))
SEED = int(os.getenv("SEED", "42"))
MIN_DELTA = float(os.getenv("MIN_DELTA", "1e-4"))
BACKBONE_LR = float(os.getenv("BACKBONE_LR", "1e-4"))
HEAD_LR = float(os.getenv("HEAD_LR", "5e-4"))
WEIGHT_DECAY = float(os.getenv("WEIGHT_DECAY", "0.01"))
LABEL_SMOOTHING = float(os.getenv("LABEL_SMOOTHING", "0.03"))
WARMUP_EPOCHS = int(os.getenv("WARMUP_EPOCHS", "1"))
PATIENCE = int(os.getenv("EARLY_STOPPING_PATIENCE", "8"))
USE_WEIGHTED_SAMPLER = os.getenv("WEIGHTED_SAMPLER", "1") == "1"
USE_CHANNELS_LAST = os.getenv("CHANNELS_LAST", "1") == "1"
USE_BF16 = os.getenv("USE_BF16", "0") == "1"
FREEZE_FIRST_EPOCH = os.getenv("FREEZE_FIRST_EPOCH", "1") == "1"
USE_DEFACTIFY = os.getenv("USE_DEFACTIFY", "1") == "1"
# Etiquetas normalizadas: 0=REAL, 1=IA.
# Modifica estos mapas según la documentación REAL de cada dataset.
HEMG_LABEL_MAP = {0: 0, 1: 1}
DEFACTIFY_LABEL_MAP = {0: 1, 1: 0}  # Supone que Defactify usa la convención opuesta.
# Selecciona "resnet50", "efficientnet_b0" o "convnext_tiny".
BACKBONE = os.getenv("BACKBONE", "resnet50").lower()
MAX_PARAMETERS = int(os.getenv("MAX_PARAMETERS", "27000000"))  # 27 millones de parámetros como máximo

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("once", category=UserWarning)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
bf16_supported = bool(use_amp and torch.cuda.is_bf16_supported())
amp_dtype = torch.bfloat16 if (USE_BF16 and bf16_supported) else torch.float16
use_scaler = use_amp and amp_dtype == torch.float16
channels_last = USE_CHANNELS_LAST and device.type == "cuda"

print(f"Dispositivo: {device}")
if use_amp:
    print(f"GPU: {torch.cuda.get_device_name(0)} | AMP={amp_dtype} | channels_last={channels_last}")
print(f"Backbone={BACKBONE}, image={IMAGE_SIZE}, batch={BATCH_SIZE}, workers={NUM_WORKERS}")
print(f"Presupuesto de entrenamiento acumulado: {MAX_RUNTIME_SECONDS / 3600:.1f} horas; "
      f"máximo de parámetros: {MAX_PARAMETERS:,}")

# ============================ DATASETS ============================
print("\nCargando datasets de Hugging Face...")

def normalize_label(value, mapping, dataset_name):
    try:
        value = int(value)
    except Exception as exc:
        raise ValueError(f"Etiqueta no numérica en {dataset_name}: {value!r}") from exc
    if value not in mapping:
        raise ValueError(f"Etiqueta {value} no definida en el mapa de {dataset_name}: {mapping}")
    return int(mapping[value])

def normalize_hf_dataset(ds, source, label_column, mapping):
    if label_column != "label":
        ds = ds.rename_column(label_column, "label")
    # Se materializa solo la columna de etiquetas, no las imágenes.
    ds = ds.map(
        lambda batch: {"label": [normalize_label(v, mapping, source) for v in batch["label"]]},
        batched=True,
        desc=f"Normalizando etiquetas {source}"
    )
    # Etiqueta de fuente útil para análisis de fugas/errores.
    ds = ds.add_column("source", [source] * len(ds))
    return ds

ds_hemg_raw = load_dataset("Hemg/AI-Generated-vs-Real-Images-Datasets", split="train")
if "label" not in ds_hemg_raw.column_names:
    raise ValueError(f"Hemg no contiene columna 'label'. Columnas: {ds_hemg_raw.column_names}")
ds_hemg = normalize_hf_dataset(ds_hemg_raw, "hemg", "label", HEMG_LABEL_MAP)
datasets_to_combine = [ds_hemg]
probabilities = [1.0]

if USE_DEFACTIFY:
    try:
        ds_def_raw = load_dataset("Rajarshi-Roy-research/Defactify_Image_Dataset", split="train")
        if "label" not in ds_def_raw.column_names:
            raise ValueError(f"Defactify no contiene 'label'. Columnas: {ds_def_raw.column_names}")
        ds_def = normalize_hf_dataset(ds_def_raw, "defactify", "label", DEFACTIFY_LABEL_MAP)
        datasets_to_combine.append(ds_def)
        probabilities = [0.5, 0.5]
        print("Defactify cargado. IMPORTANTE: confirma DEFACTIFY_LABEL_MAP con la ficha del dataset.")
    except Exception as exc:
        print(f"AVISO: no se pudo cargar Defactify; se usa Hemg. Motivo: {exc}")

if len(datasets_to_combine) == 1:
    combined = datasets_to_combine[0]
else:
    combined = interleave_datasets(
        datasets_to_combine, probabilities=probabilities, seed=SEED,
        stopping_strategy="all_exhausted"
    )

print(f"Total combinado: {len(combined):,}")
label_counts = {int(k): int(v) for k, v in zip(*np.unique(combined["label"], return_counts=True))}
print(f"Etiquetas (0=real, 1=IA): {label_counts}")
if set(label_counts) != {0, 1}:
    raise ValueError(
        f"Se esperaban las dos clases 0 y 1; encontradas {label_counts}. "
        "Revisa HEMG_LABEL_MAP y DEFACTIFY_LABEL_MAP."
    )

# Split determinista estratificado por clase. Para evitar fuga real, lo ideal es
# agrupar por ID de origen/generador/duplicados si el dataset aporta esa metadata.
labels_all = np.asarray(combined["label"], dtype=np.int64)
rng = np.random.default_rng(SEED)
train_indices, val_indices = [], []
for cls in (0, 1):
    cls_indices = np.flatnonzero(labels_all == cls)
    rng.shuffle(cls_indices)
    n_val = max(1, int(round(len(cls_indices) * VAL_FRACTION)))
    val_indices.extend(cls_indices[:n_val].tolist())
    train_indices.extend(cls_indices[n_val:].tolist())
rng.shuffle(train_indices)
rng.shuffle(val_indices)
print(f"Train: {len(train_indices):,} | Val: {len(val_indices):,}")

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

# Augmentations moderadas para no borrar artefactos globales de las imágenes.
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.90, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.06, contrast=0.06, saturation=0.06, hue=0.01),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_tf = transforms.Compose([
    transforms.Resize(int(round(IMAGE_SIZE * 256 / 224))),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class HFDataset(Dataset):
    def __init__(self, hf_dataset, indices, transform):
        self.ds = hf_dataset
        self.indices = np.asarray(indices, dtype=np.int64)
        self.transform = transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        item = self.ds[int(self.indices[idx])]
        img = item["image"]
        if not isinstance(img, Image.Image):
            img = Image.fromarray(np.asarray(img))
        # Corrige orientación EXIF cuando existe y evita imágenes paleta/alpha inconsistentes.
        img = ImageOps.exif_transpose(img).convert("RGB")
        return self.transform(img), int(item["label"])

train_ds = HFDataset(combined, train_indices, train_tf)
val_ds = HFDataset(combined, val_indices, val_tf)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

generator = torch.Generator()
generator.manual_seed(SEED)

sampler = None
shuffle = True
if USE_WEIGHTED_SAMPLER:
    train_labels = labels_all[np.asarray(train_indices)]
    counts = np.bincount(train_labels, minlength=2).astype(np.float64)
    class_weights = 1.0 / np.maximum(counts, 1.0)
    sample_weights = class_weights[train_labels]
    sampler = WeightedRandomSampler(
        torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights), replacement=True,
        generator=generator
    )
    shuffle = False

loader_kwargs = dict(
    num_workers=NUM_WORKERS,
    pin_memory=use_amp,
    worker_init_fn=seed_worker,
    generator=generator,
)
if NUM_WORKERS > 0:
    loader_kwargs["persistent_workers"] = True
    loader_kwargs["prefetch_factor"] = 2

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=shuffle, sampler=sampler,
    drop_last=False, **loader_kwargs
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, **loader_kwargs
)

# ============================ MODEL ============================
print(f"\nCreando backbone: {BACKBONE}")
if BACKBONE == "resnet50":
    weights = models.ResNet50_Weights.IMAGENET1K_V2
    model = models.resnet50(weights=weights)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(nn.Dropout(0.15), nn.Linear(in_features, 2))
elif BACKBONE == "efficientnet_b0":
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_features = model.classifier[-1].in_features
    model.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(in_features, 2))
elif BACKBONE == "convnext_tiny":
    model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, 2)
else:
    raise ValueError("BACKBONE debe ser resnet50, efficientnet_b0 o convnext_tiny")

model = model.to(device)
if channels_last:
    model = model.to(memory_format=torch.channels_last)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parámetros: {total_params:,} (límite configurado: {MAX_PARAMETERS:,})")
if total_params > MAX_PARAMETERS:
    raise RuntimeError(
        f"El modelo tiene {total_params:,} parámetros y supera el límite de {MAX_PARAMETERS:,}. "
        "Usa BACKBONE=resnet50 o efficientnet_b0, o reduce la arquitectura."
    )

def set_backbone_trainable(trainable):
    for name, param in model.named_parameters():
        is_head = (
            name.startswith("fc.") if BACKBONE == "resnet50"
            else name.startswith("classifier.")
        )
        param.requires_grad = bool(trainable or is_head)

if FREEZE_FIRST_EPOCH:
    set_backbone_trainable(False)

def make_optimizer():
    head_prefix = "fc." if BACKBONE == "resnet50" else "classifier."
    backbone_params, head_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        (head_params if name.startswith(head_prefix) else backbone_params).append(param)
    groups = []
    if backbone_params:
        groups.append({"params": backbone_params, "lr": BACKBONE_LR, "initial_lr": BACKBONE_LR})
    if head_params:
        groups.append({"params": head_params, "lr": HEAD_LR, "initial_lr": HEAD_LR})
    return optim.AdamW(groups, weight_decay=WEIGHT_DECAY)

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = make_optimizer()
scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)

# Scheduler de cosine por pasos, con warmup. Actualizado solo tras optimizador.
steps_per_epoch = max(1, len(train_loader))
total_steps = max(steps_per_epoch, EPOCHS * steps_per_epoch)
warmup_steps = max(0, WARMUP_EPOCHS * steps_per_epoch)

def apply_lr_schedule(step):
    if step < warmup_steps and warmup_steps > 0:
        factor = max(0.05, (step + 1) / warmup_steps)
    else:
        denom = max(1, total_steps - warmup_steps)
        progress = min(1.0, max(0.0, (step - warmup_steps) / denom))
        factor = 0.5 * (1.0 + math.cos(math.pi * progress))
    for group in optimizer.param_groups:
        group["lr"] = max(1e-7, group["initial_lr"] * factor)

# ============================ CHECKPOINTS / METRICS ============================
start_epoch, global_step, best_f1, epochs_without_improvement = 0, 0, -1.0, 0
elapsed_before_resume = 0.0
best_threshold = 0.5

def rng_state():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        "loader_generator": generator.get_state(),
    }

def restore_rng(state):
    if not state:
        return
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and state.get("cuda") is not None:
        torch.cuda.set_rng_state_all(state["cuda"])
    if state.get("loader_generator") is not None:
        generator.set_state(state["loader_generator"])

def save_checkpoint(epoch, step, elapsed_seconds, is_best=False):
    state = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": int(epoch),
        "global_step": int(step),
        "best_f1": float(best_f1),
        "best_threshold": float(best_threshold),
        "epochs_without_improvement": int(epochs_without_improvement),
        "elapsed_seconds": float(elapsed_seconds),
        "rng": rng_state(),
        "backbone": BACKBONE,
        "image_size": IMAGE_SIZE,
        "class_mapping": {"real": 0, "ai": 1},
        "seed": SEED,
    }
    tmp = CHECKPOINT_PATH.with_suffix(".pth.tmp")
    torch.save(state, tmp)
    os.replace(tmp, CHECKPOINT_PATH)
    if is_best:
        best_tmp = BEST_PATH.with_suffix(".pth.tmp")
        torch.save({
            "model": model.state_dict(),
            "architecture": BACKBONE,
            "image_size": IMAGE_SIZE,
            "best_f1": float(best_f1),
            "best_threshold": float(best_threshold),
            "parameters": total_params,
            "class_mapping": {"real": 0, "ai": 1},
            "mean": MEAN,
            "std": STD,
        }, best_tmp)
        os.replace(best_tmp, BEST_PATH)

def write_metrics(record):
    with METRICS_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False, allow_nan=True) + "\n")

if CHECKPOINT_PATH.exists() and os.getenv("RESUME", "1") == "1":
    print("\nReanudando desde checkpoint...")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    if ckpt.get("backbone", BACKBONE) != BACKBONE or ckpt.get("image_size", IMAGE_SIZE) != IMAGE_SIZE:
        raise ValueError("El checkpoint no coincide con BACKBONE/IMAGE_SIZE. Cambia RESUME=0 o restaura la configuración original.")
    model.load_state_dict(ckpt["model"])
    # Si la cabeza estaba congelada, reconstruye los grupos de parámetros antes de cargar optimizer.
    set_backbone_trainable(not (FREEZE_FIRST_EPOCH and int(ckpt["epoch"]) == 0))
    optimizer = make_optimizer()
    try:
        optimizer.load_state_dict(ckpt["optimizer"])
    except Exception as exc:
        raise RuntimeError(f"No se pudo restaurar el optimizador: {exc}") from exc
    if ckpt.get("scaler"):
        scaler.load_state_dict(ckpt["scaler"])
    start_epoch = int(ckpt["epoch"])
    global_step = int(ckpt["global_step"])
    best_f1 = float(ckpt["best_f1"])
    best_threshold = float(ckpt.get("best_threshold", 0.5))
    epochs_without_improvement = int(ckpt.get("epochs_without_improvement", 0))
    elapsed_before_resume = float(ckpt.get("elapsed_seconds", 0.0))
    restore_rng(ckpt.get("rng"))
    print(f"Epoch={start_epoch}, step={global_step}, mejor F1={best_f1:.4f}, tiempo acumulado={elapsed_before_resume/3600:.2f} h")

with CONFIG_PATH.open("w", encoding="utf-8") as f:
    json.dump({
        "model_name": MODEL_NAME, "backbone": BACKBONE, "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE, "num_workers": NUM_WORKERS, "seed": SEED,
        "max_runtime_seconds": MAX_RUNTIME_SECONDS, "max_parameters": MAX_PARAMETERS, "epochs": EPOCHS,
        "hemg_label_map": HEMG_LABEL_MAP, "defactify_label_map": DEFACTIFY_LABEL_MAP,
        "weighted_sampler": USE_WEIGHTED_SAMPLER, "channels_last": channels_last,
        "amp_dtype": str(amp_dtype) if use_amp else None,
    }, f, indent=2, ensure_ascii=False)

# ============================ EVALUATION ============================
def evaluate():
    model.eval()
    all_labels, all_probs = [], []
    total_loss, total_seen = 0.0, 0
    with torch.inference_mode():
        for imgs, labels in tqdm(val_loader, desc="Validación", leave=False):
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            if channels_last:
                imgs = imgs.contiguous(memory_format=torch.channels_last)
            amp_context = torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=use_amp)
            with amp_context:
                logits = model(imgs)
                loss = criterion(logits, labels)
            probs = torch.softmax(logits.float(), dim=1)
            total_loss += float(loss.item()) * labels.size(0)
            total_seen += labels.size(0)
            all_labels.extend(labels.cpu().numpy().tolist())
            all_probs.extend(probs[:, 1].cpu().numpy().tolist())
    y_true = np.asarray(all_labels, dtype=np.int64)
    p_ai = np.asarray(all_probs, dtype=np.float64)
    y_pred_05 = (p_ai >= 0.5).astype(np.int64)
    # Umbral elegido exclusivamente en validación; reporta también umbral estándar.
    thresholds = np.linspace(0.1, 0.9, 161)
    f1s = [f1_score(y_true, (p_ai >= t).astype(int), pos_label=1, zero_division=0) for t in thresholds]
    best_idx = int(np.argmax(f1s))
    threshold = float(thresholds[best_idx])
    y_pred = (p_ai >= threshold).astype(np.int64)
    metrics = {
        "loss": total_loss / max(1, total_seen),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_ai": float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "f1_real": float(f1_score(y_true, y_pred, pos_label=0, zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "precision_ai": float(precision_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "recall_ai": float(recall_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "f1_ai_at_05": float(f1_score(y_true, y_pred_05, pos_label=1, zero_division=0)),
        "threshold": threshold,
        "auc": float(roc_auc_score(y_true, p_ai)) if len(np.unique(y_true)) == 2 else float("nan"),
        "pr_auc": float(average_precision_score(y_true, p_ai)) if len(np.unique(y_true)) == 2 else float("nan"),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist(),
        "classification_report": classification_report(
            y_true, y_pred, labels=[0, 1], target_names=["real", "ia"],
            output_dict=True, zero_division=0
        ),
    }
    return metrics

# ============================ TRAIN ============================
run_started = time.monotonic()
last_checkpoint_elapsed = elapsed_before_resume
print("\n=== Inicio de entrenamiento ===")
stop_training = False
last_epoch = start_epoch

try:
    for epoch in range(start_epoch, EPOCHS):
        last_epoch = epoch
        # Tras la primera época de cabeza congelada, desbloquea el backbone.
        if FREEZE_FIRST_EPOCH and epoch >= 1:
            set_backbone_trainable(True)
            # Al cambiar los parámetros entrenables, recrear optimizer conservando LR programado de forma explícita.
            old_lrs = [g.get("lr", g["initial_lr"]) for g in optimizer.param_groups]
            optimizer = make_optimizer()
            for i, group in enumerate(optimizer.param_groups):
                group["lr"] = old_lrs[min(i, len(old_lrs)-1)]
            # Scaler conserva estado.
        model.train()
        running_loss, seen = 0.0, 0
        pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{EPOCHS}")
        for batch_idx, (imgs, labels) in pbar:
            elapsed = elapsed_before_resume + (time.monotonic() - run_started)
            # Deja margen para guardar y cerrar antes del límite.
            if elapsed >= MAX_RUNTIME_SECONDS - 300:
                stop_training = True
                break
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            if channels_last:
                imgs = imgs.contiguous(memory_format=torch.channels_last)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=use_amp):
                logits = model(imgs)
                loss = criterion(logits, labels)
            if not torch.isfinite(loss):
                print(f"AVISO: pérdida no finita en epoch={epoch+1}, batch={batch_idx}; se omite el batch.")
                optimizer.zero_grad(set_to_none=True)
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if not torch.isfinite(grad_norm):
                print(f"AVISO: gradiente no finito en epoch={epoch+1}, batch={batch_idx}; se omite el paso.")
                optimizer.zero_grad(set_to_none=True)
                scaler.update()
                continue
            apply_lr_schedule(global_step)
            scaler.step(optimizer)
            scaler.update()
            running_loss += float(loss.item()) * labels.size(0)
            seen += labels.size(0)
            global_step += 1
            pbar.set_postfix(
                loss=f"{loss.item():.4f}",
                lr=f"{optimizer.param_groups[0]['lr']:.2e}",
                grad=f"{float(grad_norm):.2f}"
            )
            if global_step % CHECKPOINT_EVERY_STEPS == 0:
                elapsed = elapsed_before_resume + (time.monotonic() - run_started)
                save_checkpoint(epoch, global_step, elapsed, is_best=False)
                print(f"\nCheckpoint guardado en step {global_step} ({elapsed/3600:.2f} h acumuladas)")
        if stop_training:
            break

        metrics = evaluate()
        best_threshold = metrics["threshold"]
        elapsed = elapsed_before_resume + (time.monotonic() - run_started)
        record = {
            "epoch": epoch + 1, "global_step": global_step,
            "train_loss": running_loss / max(1, seen), **metrics,
            "elapsed_seconds": elapsed,
        }
        write_metrics(record)
        print(
            f"\nEpoch {epoch+1}: train_loss={record['train_loss']:.4f} "
            f"val_loss={metrics['loss']:.4f} acc={metrics['accuracy']:.4f} "
            f"macroF1={metrics['macro_f1']:.4f} F1_IA={metrics['f1_ai']:.4f} "
            f"AUC={metrics['auc']:.4f} threshold={metrics['threshold']:.3f}"
        )
        print(f"Matriz de confusión [[real->real, real->IA], [IA->real, IA->IA]]: {metrics['confusion_matrix']}")

        improved = metrics["macro_f1"] > best_f1 + MIN_DELTA
        if improved:
            best_f1 = metrics["macro_f1"]
            epochs_without_improvement = 0
            save_checkpoint(epoch + 1, global_step, elapsed, is_best=True)
            print(f"Nuevo mejor macro-F1: {best_f1:.4f}")
        else:
            epochs_without_improvement += 1
            save_checkpoint(epoch + 1, global_step, elapsed, is_best=False)

        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping: {PATIENCE} épocas sin mejora.")
            break
        if elapsed >= MAX_RUNTIME_SECONDS - 300:
            stop_training = True
            break

except KeyboardInterrupt:
    print("\nInterrumpido por usuario; guardando checkpoint...")
    elapsed = elapsed_before_resume + (time.monotonic() - run_started)
    save_checkpoint(last_epoch, global_step, elapsed, is_best=False)

except Exception:
    print("\nError durante el entrenamiento. Intentando guardar checkpoint antes de salir...")
    elapsed = elapsed_before_resume + (time.monotonic() - run_started)
    try:
        save_checkpoint(last_epoch, global_step, elapsed, is_best=False)
    except Exception as save_exc:
        print(f"No se pudo guardar checkpoint: {save_exc}")
    raise

finally:
    elapsed_total = elapsed_before_resume + (time.monotonic() - run_started)
    print(f"Tiempo acumulado aproximado: {elapsed_total/3600:.2f} h")

# Exporta el mejor modelo para inferencia; si no hubo mejora nueva, usa BEST_PATH existente.
if BEST_PATH.exists():
    best_ckpt = torch.load(BEST_PATH, map_location=device, weights_only=False)
    model.load_state_dict(best_ckpt["model"])
    final_f1 = float(best_ckpt.get("best_f1", best_f1))
    final_threshold = float(best_ckpt.get("best_threshold", best_threshold))
else:
    final_f1, final_threshold = best_f1, best_threshold

final_payload = {
    "model": model.state_dict(),
    "architecture": BACKBONE,
    "image_size": IMAGE_SIZE,
    "best_macro_f1": final_f1,
    "threshold_ai": final_threshold,
    "parameters": total_params,
    "class_mapping": {"real": 0, "ai": 1},
    "mean": MEAN,
    "std": STD,
}
tmp_final = FINAL_PATH.with_suffix(".pth.tmp")
torch.save(final_payload, tmp_final)
os.replace(tmp_final, FINAL_PATH)

print("\nEntrenamiento terminado o pausado por límite de tiempo.")
print(f"Mejor checkpoint: {BEST_PATH}")
print(f"Modelo final: {FINAL_PATH}")
print(f"Métricas por época: {METRICS_PATH}")
print(f"Mejor macro-F1: {final_f1:.4f}")
print(f"Umbral sugerido para IA (ajustado en validación): {final_threshold:.3f}")
if stop_training:
    print("Se alcanzó el límite de tiempo. Ejecuta de nuevo el mismo comando para reanudar desde checkpoint.")
'''
target = Path('/kaggle/working/train.py')
target.write_text(script, encoding='utf-8')
print(f'Archivo creado: {target}. Iniciando entrenamiento automáticamente...')
subprocess.run([sys.executable, str(target)], check=True)
